# NHS GP Cloud Based Telephony · **Silver layer**

**What this notebook does, in one breath:** Bronze proved the files are exactly what NHS
England published. Silver works out what they *mean*: what each indicator code is, what
an "Unassigned" practice is, which practices make up the 86.3%, and how a practice in the
call files lines up with the registered-patient list and the national practice register.

**Rule for this layer:** every decision is made in the open. Where I change, drop or
reclassify anything, the reason is written next to the code, and the numbers are checked
before and after so nothing goes missing quietly.

Definitions come from the publication's **Supporting Information** page (snapshot in
`evidence/publication_pages/support-information/`), not from guesswork.

## Step 0 — Tools and folders

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
SILVER_DIR = PROJECT_DIR / "data" / "silver"
EVIDENCE_DIR = PROJECT_DIR / "evidence"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = sorted(p.name for p in (BRONZE_DIR / "telephony").iterdir() if p.is_dir())
ANALYSIS_MONTH = "2026-08"
pd.set_option("display.max_colwidth", 80)
print("Months in Bronze:", ", ".join(MONTHS))

Months in Bronze: 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08


## Step 1 — What the indicator codes mean

The CSVs use codes `CBT001` to `CBT007` and the metadata workbook doesn't define them.
The Supporting Information page does:

| Code | Name used here | Definition (Supporting Information) |
|---|---|---|
| CBT001 | `inbound` | All external inbound calls to the practice |
| CBT002 | `ended_in_ivr` | Caller ended the call during the automated routing stage (IVR), e.g. diverted elsewhere or hung up after a message |
| CBT003 | `answered` (by wait time) | Answered by a person in the practice, broken down by how long the caller waited |
| CBT004 | `missed` | Caller expected to speak to someone but the call ended unanswered. **Includes voicemails** |
| CBT005 | `callback_requested` | Caller asked for a virtual-queue callback while waiting |
| CBT006 | `callback_made` | The automatic callbacks made from those requests |
| CBT007 | `answered` (by duration) | Same calls as CBT003, broken down by call length. "Should be equal" to CBT003 |

The four **outcomes** are answered, ended in IVR, callback requested and missed. The page
says outright that they "may not equal the total inbound calls for some practices": a
call can fall outside the defined call flow, or be counted under more than one outcome.
So the 908-call over-count nationally in August 2026 is documented behaviour, not an
error of mine. Step 7 measures how widespread it is.

In [2]:
INDICATORS = pd.DataFrame([
    ("CBT001", "inbound", "All external inbound calls"),
    ("CBT002", "ended_in_ivr", "Ended during the IVR (automated routing) stage"),
    ("CBT003", "answered_by_wait", "Answered by practice staff, by wait-time bucket"),
    ("CBT004", "missed", "Ended unanswered while queuing or ringing; includes voicemail"),
    ("CBT005", "callback_requested", "Virtual-queue callback requested"),
    ("CBT006", "callback_made", "Automatic callback made"),
    ("CBT007", "answered_by_duration", "Answered by practice staff, by call-duration bucket"),
], columns=["indicator", "indicator_name", "definition"])
INDICATOR_NAME = dict(zip(INDICATORS.indicator, INDICATORS.indicator_name))
INDICATORS.to_csv(SILVER_DIR / "indicators.csv", index=False)
INDICATORS

,indicator,indicator_name,definition
0,CBT001,inbound,All external inbound calls
1,CBT002,ended_in_ivr,Ended during the IVR (automated routing) stage
2,CBT003,answered_by_wait,"Answered by practice staff, by wait-time bucket"
3,CBT004,missed,Ended unanswered while queuing or ringing; includes voicemail
4,CBT005,callback_requested,Virtual-queue callback requested
5,CBT006,callback_made,Automatic callback made
6,CBT007,answered_by_duration,"Answered by practice staff, by call-duration bucket"


## Step 2 — "Unassigned" is a shared phone account, not missing data

Each CSV has a block of rows where `PRACTICE_CODE = "Unassigned"`. The Supporting
Information page explains them. Suppliers report by **phone account**, and a few accounts
serve up to 10 practices on one system. When that happens NHS England can't split the
calls between the practices, so they're published under "Unassigned".

My first assumption was that these rows would also have region "Unmapped". The check
below disproved it: most Unassigned rows carry a real region, ICB and sometimes PCN. The
geography is filled in **as far down as every practice on the account shares it**. If
all the practices are in one PCN, the PCN is given. If they span PCNs but share an ICB,
only the ICB is given, and so on. Only the reverse holds strictly: every "Unmapped" row
is Unassigned.

That gives the Silver rule:
- `attribution = "practice"`: calls belong to the named practice.
- `attribution = "shared_account"`: calls are real. They count nationally, and at region,
  ICB or PCN level **wherever that code is populated**, but never at practice level.

The second table shows, for August 2026, how far down the geography the shared-account
calls can be placed.

In [3]:
checks = []
for month in MONTHS:
    b = pd.read_parquet(BRONZE_DIR / "telephony" / month / "day_time.parquet",
                        columns=["REGION_CODE", "PRACTICE_CODE"])
    unassigned, unmapped = b.PRACTICE_CODE.eq("Unassigned"), b.REGION_CODE.eq("Unmapped")
    checks.append({"month": month,
                   "unassigned_rows": int(unassigned.sum()),
                   "of_which_region_unmapped": int((unassigned & unmapped).sum()),
                   "unmapped_but_not_unassigned": int((unmapped & ~unassigned).sum())})
checks = pd.DataFrame(checks)
assert (checks.unmapped_but_not_unassigned == 0).all(), "an Unmapped row is attributed to a practice"

b = pd.read_parquet(BRONZE_DIR / "telephony" / ANALYSIS_MONTH / "day_time.parquet")
u = b[b.PRACTICE_CODE.eq("Unassigned") & b.Indicator.eq("CBT001")]
u_calls = u.Value.astype(float)
depth = pd.DataFrame({
    "level": ["region", "ICB", "sub-ICB", "PCN"],
    "share_of_shared_account_calls_placed": [
        round(u_calls[u[c].ne("Unmapped")].sum() / u_calls.sum(), 3)
        for c in ["REGION_CODE", "ICB_CODE", "SUB_ICB_LOCATION_CODE", "PCN_CODE"]],
})
print(f"{ANALYSIS_MONTH}: shared-account inbound calls = {int(u_calls.sum()):,} "
      f"({u_calls.sum() / b.loc[b.Indicator.eq('CBT001'), 'Value'].astype(float).sum():.1%} of all inbound)")
del b, u
print(checks.to_string())
depth

2026-08: shared-account inbound calls = 1,749,178 (6.4% of all inbound)
      month  unassigned_rows  of_which_region_unmapped  unmapped_but_not_unassigned
0   2025-10            85108                      1484                            0
1   2025-11            94320                      1379                            0
2   2025-12           105534                      1503                            0
3   2026-01            98383                      1543                            0
4   2026-02            90674                      1378                            0
5   2026-03           101320                      1535                            0
6   2026-04            93713                      1521                            0
7   2026-05            90252                      1507                            0
8   2026-06            96294                      1505                            0
9   2026-07            98061                      1579                            0
10  

,level,share_of_shared_account_calls_placed
0,region,0.906
1,ICB,0.849
2,sub-ICB,0.746
3,PCN,0.393


## Step 3 — Type the call files

The same treatment for all three call files and every month:

- **Columns renamed** to snake_case, and the detail column gets a meaningful name:
  `time_band`, `duration_bucket` or `wait_bucket`.
- **`Value` becomes `calls`, an integer.** The metadata calls it a float, but it's a
  count, so I check that every value is a whole, non-negative number before casting.
  Anything else stops the run rather than being rounded quietly.
- **`date`** is parsed and checked to fall inside its publication month. `weekday` is
  added because the Monday question depends on it.
- **Practice codes** are trimmed and upper-cased (the join rule in `docs/sources.md`).
  The cell reports how many changed. If none did, the rule is cheap insurance.
- **Buckets** like `121_180` or `>300` get numeric bounds in seconds (`lower_s`,
  `upper_s`), so Gold can group them without parsing strings again.

Output: one Parquet file per month under `data/silver/calls_<kind>/`.

In [4]:
KIND_DETAIL = {"day_time": "time_band", "durations": "duration_bucket", "calls_answered": "wait_bucket"}
RENAME = {"Date": "date", "REGION_CODE": "region_code", "ICB_CODE": "icb_code",
          "SUB_ICB_LOCATION_CODE": "sub_icb_code", "PCN_CODE": "pcn_code",
          "PRACTICE_CODE": "practice_code", "Indicator": "indicator",
          "Time_Category": "time_band", "Duration": "duration_bucket",
          "Wait_Time": "wait_bucket", "Core_Category": "core_category"}


def bucket_bounds(bucket: pd.Series) -> pd.DataFrame:
    s = bucket.astype(str)
    lower = pd.Series(np.nan, index=s.index)
    upper = pd.Series(np.nan, index=s.index)
    rng = s.str.extract(r"^(\d+)_(\d+)$").astype(float)
    lower, upper = lower.fillna(rng[0]), upper.fillna(rng[1])
    zero = s.eq("0")
    lower[zero], upper[zero] = 0, 0
    over = s.str.extract(r"^>(\d+)$")[0].astype(float)
    lower = lower.fillna(over + 1)
    unknown = ~(s.eq("TOTAL") | lower.notna())
    assert not unknown.any(), f"Unrecognised buckets: {sorted(s[unknown].unique())}"
    return pd.DataFrame({"lower_s": lower.astype("Int64"), "upper_s": upper.astype("Int64")})


def silver_calls(kind: str, month: str) -> tuple[pd.DataFrame, dict]:
    b = pd.read_parquet(BRONZE_DIR / "telephony" / month / f"{kind}.parquet")
    s = b[[c for c in b.columns if not c.startswith("_")]].rename(columns=RENAME)

    calls = pd.to_numeric(s.pop("Value"), errors="coerce")
    assert calls.notna().all(), f"{kind} {month}: non-numeric Value"
    assert (calls >= 0).all() and (calls == calls.round()).all(), f"{kind} {month}: Value not a whole count"
    s["calls"] = calls.astype("int64")

    s["date"] = pd.to_datetime(s["date"], format="ISO8601")
    assert (s["date"].dt.strftime("%Y-%m") == month).all(), f"{kind} {month}: date outside month"
    s.insert(1, "weekday", s["date"].dt.day_name())

    shared = s["practice_code"].eq("Unassigned")
    cleaned = s["practice_code"].where(shared, s["practice_code"].str.strip().str.upper())
    changed = int((cleaned != s["practice_code"]).sum())
    s["practice_code"] = cleaned
    s["attribution"] = np.where(shared, "shared_account", "practice")
    s["indicator_name"] = s["indicator"].map(INDICATOR_NAME)
    assert s["indicator_name"].notna().all(), f"{kind} {month}: unknown indicator"

    detail = KIND_DETAIL[kind]
    if detail == "time_band":
        s["time_band"] = s["time_band"].str.replace(" ", "", regex=False)
    else:
        s = pd.concat([s, bucket_bounds(s[detail])], axis=1)

    s.insert(0, "month", month)
    for c in s.columns:
        if s[c].dtype == object:
            s[c] = s[c].astype("category")

    out_dir = SILVER_DIR / f"calls_{kind}"
    out_dir.mkdir(exist_ok=True)
    s.to_parquet(out_dir / f"{month}.parquet", index=False)
    return s, {"kind": kind, "month": month, "bronze_rows": len(b), "silver_rows": len(s),
               "value_sum_all_indicators": int(s["calls"].sum()), "practice_codes_changed_by_trim": changed}


log, national = [], []
for month in MONTHS:
    for kind in KIND_DETAIL:
        if not (BRONZE_DIR / "telephony" / month / f"{kind}.parquet").exists():
            continue
        s, entry = silver_calls(kind, month)
        log.append(entry)
        national.append(s.groupby(["month", "indicator"], observed=True)["calls"].sum()
                         .rename(kind).reset_index())
        del s

silver_log = pd.DataFrame(log)
assert (silver_log.bronze_rows == silver_log.silver_rows).all(), "rows lost between Bronze and Silver"
print("Practice codes changed by trim/upper-case:", silver_log.practice_codes_changed_by_trim.sum())
silver_log

Practice codes changed by trim/upper-case: 0


,kind,month,bronze_rows,silver_rows,value_sum_all_indicators,practice_codes_changed_by_trim
0,day_time,2025-10,3714703,3714703,82387291,0
1,durations,2025-10,2733623,2733623,82387291,0
2,day_time,2025-11,3536837,3536837,77034130,0
3,durations,2025-11,2583806,2583806,77034130,0
4,day_time,2025-12,3706340,3706340,78808181,0
5,durations,2025-12,2689838,2689838,78808181,0
6,day_time,2026-01,3811913,3811913,82996852,0
7,durations,2026-01,2744540,2744540,82996852,0
8,day_time,2026-02,3540227,3540227,76728992,0
9,durations,2026-02,2588234,2588234,76728992,0


## Step 4 — Do the three call files agree with each other?

The same indicator turns up in more than one file. `CBT001` is in both the day-and-time
file and the durations file (as a `TOTAL`), and `CBT003` is in the durations file and,
from June 2026, the calls-answered file. If two files give different national totals
for the same indicator and month, one of them is incomplete, and Gold needs to know
which to trust.

The table shows national totals per indicator, month and file. Where an indicator
appears in two files, `max_diff` is the gap between them.

In [5]:
by_kind = [pd.concat([f for f in national if kind in f.columns], ignore_index=True) for kind in KIND_DETAIL]
nat = by_kind[0]
for frame in by_kind[1:]:
    nat = nat.merge(frame, on=["month", "indicator"], how="outer")
present = nat[list(KIND_DETAIL)]
nat["max_diff"] = (present.max(axis=1) - present.min(axis=1)).where(present.notna().sum(axis=1) > 1)
nat["indicator_name"] = nat["indicator"].map(INDICATOR_NAME)
nat.to_csv(SILVER_DIR / "national_by_indicator_and_file.csv", index=False)
print("Indicator/month pairs where files disagree:", int((nat.max_diff > 0).sum()))
nat[nat.month == ANALYSIS_MONTH]

Indicator/month pairs where files disagree: 0


,month,indicator,day_time,durations,calls_answered,max_diff,indicator_name
70,2026-08,CBT001,27160966,27160966,NaN,0.0,inbound
71,2026-08,CBT002,6866642,6866642,NaN,0.0,ended_in_ivr
72,2026-08,CBT003,15675776,15675776,15675776.0,0.0,answered_by_wait
73,2026-08,CBT004,2817449,2817449,NaN,0.0,missed
74,2026-08,CBT005,1802007,1802007,NaN,0.0,callback_requested
75,2026-08,CBT006,1794190,1794190,NaN,0.0,callback_made
76,2026-08,CBT007,15675776,15675776,NaN,0.0,answered_by_duration


## Step 5 — The practice list behind the 86.3%

Each month comes with a participation workbook listing every open, active practice and
three Yes/No flags: did it agree to take part, was its data received and included, and
was it put in a shared ("unallocated") account. That list is the denominator for
coverage, so it's the backbone of the practice dimension.

The workbook is formatted for reading, with notes above the table. So I find the header
row by its text, check the column headings are the ones I expect, and keep only rows
whose code has the ODS practice format (a letter followed by five digits).

In [6]:
PARTICIPATION_COLS = ["month_published", "practice_code", "practice_name", "pcn_code", "pcn_name",
                      "sub_icb_code", "sub_icb_name", "icb_code", "icb_name", "region_code",
                      "region_name", "agreed", "included", "shared_account", "account_practice_codes"]
EXPECTED_HEADINGS = ["Month", "GP Code", "GP Name", "PCN Code", "PCN Name", "Sub ICB Code",
                     "Sub ICB Name", "ICB Code", "ICB Name", "Region Code", "Region Name",
                     "Agreed to Participate", "Data received / Passed", "Practice Code has been un",
                     "List of open and act"]  # Oct 2025 heading reads "actice"
ODS_PRACTICE = re.compile(r"^[A-Z]\d{5}$")
HEADING_GAPS = []


def read_participation(month: str) -> pd.DataFrame:
    path = next((BRONZE_DIR / "telephony" / month).glob("CBT-participation*.xlsx"))
    raw = pd.read_excel(path, sheet_name="Table 1", header=None, dtype=str)
    hdr = raw.index[raw[0].astype(str).str.strip().eq("Month")][0]
    headings = raw.iloc[hdr, :15].astype(str).tolist()
    for got, want in zip(headings, EXPECTED_HEADINGS):
        if got == "nan":  # Nov 2025 leaves the "included" heading blank; the Yes/No check below still applies
            HEADING_GAPS.append((month, want))
            continue
        assert want.lower() in got.lower(), f"{month}: heading {got!r} - expected {want!r}"
    body = raw.iloc[hdr + 1:, :15].copy()
    body.columns = PARTICIPATION_COLS
    body = body[body.practice_code.notna()]
    body["practice_code"] = body.practice_code.str.strip().str.upper()
    odd = body[~body.practice_code.str.match(ODS_PRACTICE)]
    assert odd.empty, f"{month}: non-ODS codes {odd.practice_code.tolist()[:5]}"
    for flag in ["agreed", "included", "shared_account"]:
        assert body[flag].isin(["Yes", "No"]).all(), f"{month}: {flag} not Yes/No"
        body[flag] = body[flag].eq("Yes")
    body["account_practice_codes"] = body.account_practice_codes.fillna("")
    body["account_size"] = body.account_practice_codes.str.split(",").map(lambda x: len([c for c in x if c]))
    body.insert(0, "month", month)
    return body.drop(columns="month_published")


participation = pd.concat([read_participation(m) for m in MONTHS], ignore_index=True)
assert not participation.duplicated(["month", "practice_code"]).any()
print("Blank headings accepted (column checked by content instead):", HEADING_GAPS)

summary = participation.groupby("month").agg(
    open_active=("practice_code", "size"),
    agreed=("agreed", "sum"),
    included=("included", "sum"),
    included_shared_account=("shared_account", lambda s: int((s & participation.loc[s.index, "included"]).sum())),
)
summary

Blank headings accepted (column checked by content instead): [('2025-11', 'Data received / Passed')]


,open_active,agreed,included,included_shared_account
month,,,,
2025-10,6220,6152,4649,274
2025-11,6208,6141,5015,324
2025-12,6211,6146,5093,354
2026-01,6209,6142,5109,350
2026-02,6208,6144,5112,349
2026-03,6207,6143,5245,352
2026-04,6205,6142,5292,347
2026-05,6195,6132,5290,342
2026-06,6186,6123,5292,341


### Which included practices actually have their own rows in the call files?

This is the 4,990-vs-5,327 question from Bronze profiling. My expectation: an included
practice on its own account appears under its own code, and an included practice on a
shared account doesn't, because its calls sit in the Unassigned block. The cell tests
that, month by month, in both directions.

In [7]:
def csv_practices(month: str) -> set:
    s = pd.read_parquet(SILVER_DIR / "calls_day_time" / f"{month}.parquet", columns=["practice_code", "attribution"])
    return set(s.loc[s.attribution.eq("practice"), "practice_code"].astype(str))


rows = []
for month in MONTHS:
    p = participation[participation.month == month]
    in_csv = csv_practices(month)
    own = set(p.loc[p.included & ~p.shared_account, "practice_code"])
    shared = set(p.loc[p.included & p.shared_account, "practice_code"])
    rows.append({
        "month": month,
        "included_own_account": len(own),
        "included_shared_account": len(shared),
        "csv_practice_codes": len(in_csv),
        "own_account_missing_from_csv": len(own - in_csv),
        "shared_but_published_individually": len(shared & in_csv),
        "csv_codes_not_included": len(in_csv - own - shared),
    })
    participation.loc[participation.month == month, "has_own_call_rows"] = \
        participation.loc[participation.month == month, "practice_code"].isin(in_csv)

practice_match = pd.DataFrame(rows)
practice_match

,month,included_own_account,included_shared_account,csv_practice_codes,own_account_missing_from_csv,shared_but_published_individually,csv_codes_not_included
0,2025-10,4375,274,4376,0,1,0
1,2025-11,4691,324,4691,0,0,0
2,2025-12,4739,354,4739,0,0,0
3,2026-01,4759,350,4766,0,7,0
4,2026-02,4763,349,4771,0,8,0
5,2026-03,4893,352,4901,0,8,0
6,2026-04,4945,347,4953,0,8,0
7,2026-05,4948,342,4957,0,9,0
8,2026-06,4951,341,4960,0,9,0
9,2026-07,4972,348,4982,0,10,0


**Reading this table (August 2026):** 5,327 included = 4,977 practices on their own
account + 350 on shared accounts. Every one of the 4,977 has its own rows. The 350 don't,
except for 12 that are flagged "shared" yet still published under their own code. In
every one of those 12, the account list holds only the practice itself. Note 3 of the
participation workbook explains why: other practices on the account were excluded from
the publication, so they aren't listed. I keep the 12 as practice-attributed (that's how
NHS England published them) and flag them with `shared_but_published_individually`.

So the 4,989 practice codes in the CSV (plus the Unassigned block) = 4,977 + 12. Nothing
is missing.

In [8]:
participation["has_own_call_rows"] = participation.has_own_call_rows.astype(bool)
participation["shared_but_published_individually"] = participation.shared_account & participation.has_own_call_rows
participation["attribution"] = np.select(
    [~participation.included,
     participation.has_own_call_rows,
     participation.shared_account],
    ["not_included", "practice", "shared_account"],
    default="included_but_no_rows",
)
print(participation[participation.month == ANALYSIS_MONTH].attribution.value_counts().to_string())
participation.to_parquet(SILVER_DIR / "practice_participation.parquet", index=False)

attribution
practice          4989
not_included       844
shared_account     338


## Step 6 — Registered patients and the ODS practice register

**ODS `epraccur`.** Bronze left the 27 columns unnamed because the file didn't match the
legacy layout I expected. The current ODS Data Search & Export reference catalogue gives
the layout (snapshot in `evidence/reference/ods_reference_data_catalogue_2026-10-02.html`).
Two changes from the legacy file matter here:

- Column 13 is now the full status name (`ACTIVE` / `INACTIVE` / `DORMANT`), not `A`/`C`/`D`.
- Column 26 is now the ODS **role code**, not the old numeric "prescribing setting". GP
  practices are `RO76` (formerly setting 4). Everything else in the file is another
  kind of prescribing cost centre: walk-in centres, out-of-hours, prisons, and so on.

**Registered patients (1 August 2026).** The practice totals and the PCN/ICB/region
mapping should be one row per practice and match one-to-one. The cell checks that before
joining them.

In [9]:
EPRACCUR_COLS = [
    "practice_code", "name", "nhser_code", "icb_code", "address_1", "address_2", "address_3",
    "town", "county", "postcode", "open_date", "close_date", "status", "org_sub_type",
    "commissioner_sub_icb", "join_provider_date", "left_provider_date", "telephone",
    "null_19", "null_20", "null_21", "amended_record_indicator", "null_23",
    "provider_purchaser", "null_25", "role_code", "null_27",
]
e = pd.read_parquet(next((BRONZE_DIR / "ods").glob("epraccur_*/epraccur.parquet")))
snapshot = e["_snapshot"].iloc[0]
e = e[[f"col_{i:02d}" for i in range(1, 28)]]
e.columns = EPRACCUR_COLS
null_cols = [c for c in EPRACCUR_COLS if c.startswith("null_")]
assert (e[null_cols] == "").all().all(), "a column the spec says is NULL has data"
register = e.drop(columns=null_cols + ["county", "amended_record_indicator"])
for c in ["open_date", "close_date", "join_provider_date", "left_provider_date"]:
    register[c] = pd.to_datetime(register[c].replace("", None), format="%Y%m%d")
register["is_gp_practice"] = register.role_code.str.split("|").map(lambda r: "RO76" in r)
register["snapshot_date"] = pd.Timestamp(snapshot)
register.to_parquet(SILVER_DIR / "practice_register.parquet", index=False)

gp = register[register.is_gp_practice]
print(f"epraccur {snapshot}: {len(register):,} prescribing cost centres, {len(gp):,} GP practices (RO76)")
print(gp.status.value_counts().to_string())

reg_dir = BRONZE_DIR / "registered_patients" / ANALYSIS_MONTH
totals = pd.read_parquet(reg_dir / "gp-reg-pat-prac-all.parquet")
mapping = pd.read_parquet(reg_dir / "gp-reg-pat-prac-map.parquet")
assert totals.CODE.is_unique and mapping.PRACTICE_CODE.is_unique
assert (totals.SEX == "ALL").all() and (totals.AGE == "ALL").all()
assert set(totals.CODE) == set(mapping.PRACTICE_CODE), "totals and mapping cover different practices"
registered = (totals[["CODE", "EXTRACT_DATE", "NUMBER_OF_PATIENTS"]]
              .rename(columns={"CODE": "practice_code", "EXTRACT_DATE": "extract_date",
                               "NUMBER_OF_PATIENTS": "registered_patients"})
              .merge(mapping.drop(columns=["PUBLICATION", "EXTRACT_DATE"])
                            .rename(columns=str.lower), on="practice_code", how="inner"))
registered["registered_patients"] = registered.registered_patients.astype("int64")
registered["extract_date"] = pd.to_datetime(registered.extract_date)
registered = registered.rename(columns={"supplier_name": "clinical_system_supplier"})
registered.to_parquet(SILVER_DIR / "registered_patients.parquet", index=False)
print(f"Registered patients {registered.extract_date.iloc[0]:%d %b %Y}: {len(registered):,} practices, "
      f"{registered.registered_patients.sum():,} patients")

epraccur 2026-10-02: 15,659 prescribing cost centres, 8,193 GP practices (RO76)
status
ACTIVE      6562
INACTIVE    1568
DORMANT       63
Registered patients 01 Aug 2026: 6,129 practices, 63,237,908 patients


`SUPPLIER_NAME` in the mapping file is renamed to `clinical_system_supplier` on purpose.
It's the GP clinical system (EMIS, TPP), **not** the telephony supplier. Calling it
"supplier" would invite exactly the proxy that `docs/sources.md` rules out.

### Practice deprivation (IMD 2025)

Fingertips indicator 94240 gives each GP practice a deprivation score on the IMD 2025
scale, where **higher means more deprived**. I checked the direction against the data:
the highest-scoring sub-ICBs are Blackpool, Oldham, Manchester and Bradford, and the lowest
are Surrey, York and Oxfordshire.

**What the score represents.** Fingertips' metadata for 94240 gives only the name and
source (MHCLG). The usual practice-level method weights the IMD scores of the
neighbourhoods (LSOAs) **where a practice's registered patients live**, rather than using
the practice's own postcode. NHS England documents this method (snapshot in
`evidence/reference/nhse_practice_imd_method_2026-10-02.html`). I haven't been able to
confirm Fingertips uses it for 94240, so it stays an open question in `docs/sources.md`.

Silver keeps the score as published. Grouping practices into quintiles is an analytical
choice, so that happens in Gold.

In [10]:
fp = pd.read_parquet(next((BRONZE_DIR / "fingertips").glob("imd2025_gp_*/indicator_94240_gp.parquet")))
assert fp["Indicator ID"].eq("94240").all() and fp["Time period"].eq("2025").all()
england_score = float(fp.loc[fp["Area Type"].eq("England"), "Value"].iloc[0])
imd = (fp[fp["Area Type"].eq("GPs")][["Area Code", "Area Name", "Value"]]
       .rename(columns={"Area Code": "practice_code", "Area Name": "fingertips_name", "Value": "imd2025_score"}))
imd["imd2025_score"] = pd.to_numeric(imd.imd2025_score, errors="raise")
assert imd.practice_code.is_unique
imd["snapshot_date"] = pd.Timestamp(fp["_snapshot"].iloc[0])
imd.to_parquet(SILVER_DIR / "practice_imd2025.parquet", index=False)
print(f"Fingertips IMD 2025: {len(imd):,} GP practices; England score {england_score:.2f}; "
      f"practice range {imd.imd2025_score.min():.1f} to {imd.imd2025_score.max():.1f}")

Fingertips IMD 2025: 6,143 GP practices; England score 21.81; practice range 3.9 to 65.4


### Joining the three practice lists (August 2026)

The participation list (6,171 open and active practices) is the spine. Every other list
is joined to it and unmatched codes are reported in **both** directions.

In [11]:
aug = participation[participation.month == ANALYSIS_MONTH]
P = set(aug.practice_code)
R = set(registered.practice_code)
D = set(imd.practice_code)
G_active = set(gp.loc[gp.status.eq("ACTIVE") & gp.nhser_code.str.startswith("Y"), "practice_code"])

join_report = pd.DataFrame([
    ("participation list (open & active, Aug 2026)", len(P), ""),
    ("  ...not in registered-patients file", len(P - R), ", ".join(sorted(P - R)[:6]) + (" ..." if len(P - R) > 6 else "")),
    ("  ...not ACTIVE GP practice in epraccur (2 Oct)", len(P - G_active), ", ".join(sorted(P - G_active))),
    ("registered-patients file", len(R), ""),
    ("  ...not in participation list", len(R - P), ", ".join(sorted(R - P))),
    ("ACTIVE GP practices in England, epraccur (2 Oct)", len(G_active), ""),
    ("  ...not in participation list", len(G_active - P), ", ".join(sorted(G_active - P))),
    ("participation list ...without a Fingertips IMD 2025 score", len(P - D), ", ".join(sorted(P - D)[:6]) + (" ..." if len(P - D) > 6 else "")),
    ("Fingertips IMD 2025 practices", len(D), ""),
    ("  ...not in participation list", len(D - P), ", ".join(sorted(D - P)[:6]) + (" ..." if len(D - P) > 6 else "")),
], columns=["set", "count", "codes"])
print("epraccur status of the participation practices it doesn't list as active:",
      gp[gp.practice_code.isin(P - G_active)].status.value_counts().to_dict())
join_report

epraccur status of the participation practices it doesn't list as active: {'DORMANT': 8}


,set,count,codes
0,"participation list (open & active, Aug 2026)",6171,
1,...not in registered-patients file,42,"B86009, E83666, F84750, H84053, K84008, N84043 ..."
2,...not ACTIVE GP practice in epraccur (2 Oct),8,"B86009, F81746, G82027, G83015, H84053, H85637, K84008, Y03303"
3,registered-patients file,6129,
4,...not in participation list,0,
5,"ACTIVE GP practices in England, epraccur (2 Oct)",6165,
6,...not in participation list,2,"W93019, Y08402"
7,participation list ...without a Fingertips IMD 2025 score,77,"C84720, E83666, E87007, E87711, E87772, F84750 ..."
8,Fingertips IMD 2025 practices,6143,
9,...not in participation list,49,"A81065, A82613, B81006, B81052, B81616, B81642 ..."


**What the join shows:**

- **epraccur reproduces the 6,171 denominator to within timing.** The participation list
  is 6,163 practices that are still active on 2 October plus 8 that have since gone
  dormant. epraccur adds 2 active practices that aren't on the list. The epraccur
  snapshot is two months later than the call month, so this is the expected scale of
  drift, and the published 6,171 stays the denominator.
- **42 participating practices have no registered-patient row** for 1 August. Their
  patient counts are unknown here, not zero. Step 8 checks how much that matters against
  the published patient totals.
- **77 participating practices have no deprivation score.** The next cell shows they're
  mostly practices not included in the call data, so they barely affect a
  coverage-by-deprivation cut. Gold reports them as their own "no score" group rather
  than dropping them.
- **49 Fingertips practices aren't on the August list.** All 49 are dormant or inactive in
  epraccur: 46 dropped off the participation list during the series, and 3 were never on
  it. Fingertips' practice list simply predates their closure.

In [12]:
practice_dim = (aug.drop(columns=["month"])
                .merge(registered[["practice_code", "registered_patients", "clinical_system_supplier",
                                   "comm_region_code", "icb_code"]]
                       .rename(columns={"icb_code": "icb_code_registered", "comm_region_code": "region_code_registered"}),
                       on="practice_code", how="left", indicator="in_registered")
                .merge(register[["practice_code", "status", "postcode", "open_date", "close_date"]]
                       .rename(columns={"status": "ods_status", "postcode": "ods_postcode"}),
                       on="practice_code", how="left")
                .merge(imd[["practice_code", "imd2025_score"]], on="practice_code", how="left"))
practice_dim["in_registered"] = practice_dim.in_registered.eq("both")
practice_dim.insert(0, "month", ANALYSIS_MONTH)
geo_mismatch = practice_dim.in_registered & (practice_dim.icb_code != practice_dim.icb_code_registered)
print("ICB differs between participation list and registered-patients mapping:", int(geo_mismatch.sum()))
no_score = practice_dim[practice_dim.imd2025_score.isna()]
print("Participating practices without an IMD score, by attribution:", no_score.attribution.value_counts().to_dict(),
      f"| registered patients: {int(no_score.registered_patients.fillna(0).sum()):,}")
practice_dim.to_parquet(SILVER_DIR / f"practice_dim_{ANALYSIS_MONTH}.parquet", index=False)
join_report.to_csv(EVIDENCE_DIR / f"silver_practice_join_{ANALYSIS_MONTH}.csv", index=False)
practice_dim.attribution.value_counts()

ICB differs between participation list and registered-patients mapping: 0
Participating practices without an IMD score, by attribution: {'not_included': 67, 'practice': 6, 'shared_account': 4} | registered patients: 12,197


attribution
practice          4989
not_included       844
shared_account     338
Name: count, dtype: int64

## Step 7 — How often do the four outcomes fail to add up?

The Supporting Information page says it happens for "a large number of practices". A
national over-count of 908 calls could hide much larger gaps that cancel each other out,
so I measure it at practice level: answered (CBT003) + ended in IVR (CBT002) + callback
requested (CBT005) + missed (CBT004), against inbound (CBT001).

I also compare CBT003 with CBT007. Both count answered calls, one split by wait time and
the other by call length. The specification says they should be equal.

Source: the durations file, which has every indicator for every month.

In [13]:
def outcome_check(month: str) -> pd.DataFrame:
    d = pd.read_parquet(SILVER_DIR / "calls_durations" / f"{month}.parquet",
                        columns=["practice_code", "attribution", "indicator", "calls"])
    d = d[d.attribution.eq("practice")]
    w = d.groupby(["practice_code", "indicator"], observed=True)["calls"].sum().unstack(fill_value=0)
    w = w.reindex(columns=INDICATORS.indicator, fill_value=0)
    w["outcomes"] = w[["CBT002", "CBT003", "CBT004", "CBT005"]].sum(axis=1)
    w["gap"] = w["outcomes"] - w["CBT001"]
    w["month"] = month
    return w.reset_index()


oc = pd.concat([outcome_check(m) for m in MONTHS], ignore_index=True)
tol = 0.01
oc_summary = oc.groupby("month").apply(lambda t: pd.Series({
    "practices": len(t),
    "exact_match": int((t.gap == 0).sum()),
    "outcomes_over_inbound_by_>1%": int((t.gap > tol * t.CBT001).sum()),
    "outcomes_under_inbound_by_>1%": int((t.gap < -tol * t.CBT001).sum()),
    "net_gap_calls": int(t.gap.sum()),
    "gross_gap_calls": int(t.gap.abs().sum()),
    "cbt003_ne_cbt007": int((t.CBT003 != t.CBT007).sum()),
}), include_groups=False)
oc.to_parquet(SILVER_DIR / "practice_outcome_check.parquet", index=False)
oc_summary

,practices,exact_match,outcomes_over_inbound_by_>1%,outcomes_under_inbound_by_>1%,net_gap_calls,gross_gap_calls,cbt003_ne_cbt007
month,,,,,,,
2025-10,4376,3368,6,61,-20288,23874,0
2025-11,4691,4566,4,27,-5182,6488,1
2025-12,4739,4635,5,1,1156,2044,1
2026-01,4766,4654,5,3,691,2961,0
2026-02,4771,4657,6,2,1081,3027,0
2026-03,4901,4784,7,1,1721,3051,1
2026-04,4953,4844,6,2,1357,2899,0
2026-05,4957,4838,5,1,918,2490,0
2026-06,4960,4848,4,0,1132,2188,0


**What this shows:** in August 2026, 4,881 of 4,989 practice-attributed practices (98%)
have outcomes that add up exactly to inbound calls. The national +908 is a net figure:
1,720 calls of gross mismatch across about 100 practices, most of them small. October
2025 was much worse, with about 1,000 practices off and outcomes **under** inbound by
20,288 calls net. So the "large number of practices" warning on the Supporting
Information page describes the early months better than the current ones. CBT003 and
CBT007 agree for every practice in every month except three single-practice blips.

For Gold, that means the outcome shares can be quoted against inbound with a footnote:
in August 2026 they over-count by 908 calls in 27.2 million (0.003%).

## Step 8 — Parse the published headline tables

The reconciliation gate in Gold checks my figures against NHS England's own, so the
published figures need to be data too. Two tables do this:

- **Table 1** (coverage, outcomes, wait times, durations): a time series. Every month's
  workbook carries its own copy, so parsing all eleven also shows **whether earlier
  months were revised later**.
- **Table 2**: national inbound calls by date and time band, which is where the Monday
  08:00–10:00 figure comes from.

Values are stored exactly as published: no rounding, no relabelling. Where a label looks
wrong, I flag it beside the value. I don't change it.

In [14]:
def summary_path(month: str) -> Path:
    return next((BRONZE_DIR / "telephony" / month).glob("*Publication Summary*.xlsx"))


def parse_table1(month: str) -> pd.DataFrame:
    raw = pd.read_excel(summary_path(month), sheet_name="Table 1", header=None)
    is_date = raw.map(lambda v: isinstance(v, pd.Timestamp) or hasattr(v, "year"))
    date_row = is_date.sum(axis=1).idxmax()
    date_cols = [c for c in raw.columns if is_date.loc[date_row, c]]
    first = date_cols[0]
    out, group = [], None
    for i in range(date_row + 1, len(raw)):
        row = raw.iloc[i]
        labels = [str(v).strip() for v in row[:first] if pd.notna(v) and str(v).strip()]
        values = pd.to_numeric(row[date_cols], errors="coerce")
        if str(row.iloc[0]).startswith("Notes"):
            break
        if not labels or values.isna().all():
            continue
        if len(labels) == 2:
            group = labels[0]
        for col, v in zip(date_cols, values):
            out.append({"publication_month": month,
                        "reference_month": pd.Timestamp(raw.loc[date_row, col]).strftime("%Y-%m"),
                        "group": group, "measure": labels[-1], "value": v})
    return pd.DataFrame(out)


def parse_table2(month: str) -> pd.DataFrame:
    raw = pd.read_excel(summary_path(month), sheet_name="Table 2", header=None)
    hdr = raw.index[raw.apply(lambda r: r.astype(str).str.contains("Total Count of Inbound").any(), axis=1)][0]
    heads = raw.iloc[hdr].tolist()
    total_col = heads.index(next(h for h in heads if isinstance(h, str) and "Total Count" in h))
    band_cols = [c for c in range(total_col + 1, len(heads)) if isinstance(heads[c], str)]
    rows = []
    for i in range(hdr + 1, len(raw)):
        date = raw.iloc[i, total_col - 1]
        if not hasattr(date, "year"):
            if rows:
                break
            continue
        for c in band_cols:
            rows.append({"publication_month": month, "date": pd.Timestamp(date),
                         "time_band": heads[c].replace(" ", ""), "calls": raw.iloc[i, c]})
        rows.append({"publication_month": month, "date": pd.Timestamp(date),
                     "time_band": "TOTAL", "calls": raw.iloc[i, total_col]})
    return pd.DataFrame(rows)


table1 = pd.concat([parse_table1(m) for m in MONTHS], ignore_index=True)
table2 = pd.concat([parse_table2(m) for m in MONTHS], ignore_index=True)
table2["calls"] = table2.calls.astype("int64")
table2["weekday"] = table2.date.dt.day_name()
print(f"Table 1: {len(table1):,} values, {table1.measure.nunique()} measures, from {table1.publication_month.nunique()} workbooks")
print(f"Table 2: {table2.date.nunique()} dates")
table1[(table1.publication_month == ANALYSIS_MONTH) & (table1.reference_month == ANALYSIS_MONTH)][["group", "measure", "value"]]

Table 1: 7,382 values, 44 measures, from 11 workbooks
Table 2: 335 dates


,group,measure,value
6410,Coverage,Open active practices,6.171000e+03
6428,Coverage,Count of practices included,5.327000e+03
6446,Coverage,Practice coverage [1],8.630000e-01
6464,Coverage,Registered patients at open active practices,6.325743e+07
6482,Coverage,Registered patients at included practices,5.533352e+07
6500,Coverage,Patient coverage,8.750000e-01
6518,Working Days,Number of working weekdays,2.000000e+01
6536,Inbound calls,Total Inbound Calls,2.716097e+07
6554,Inbound calls,"Rate of inbound calls per 1,000 patients registered at practices included",4.909000e+02
6572,Calls dealt with,Calls answered [2],1.567578e+07


### Were earlier months revised?

Each workbook republishes the whole series. For every (reference month, group, measure)
published more than once, I check whether the value changed between editions. The group
matters: "Calls answered within 1 minute" appears three times in Table 1 (any time, core
hours, 8–10am).

In [15]:
revisions = (table1.dropna(subset=["value"])
             .groupby(["reference_month", "group", "measure"])["value"]
             .agg(editions="size", distinct=lambda v: v.round(6).nunique(), first="first", last="last")
             .reset_index())
assert revisions.editions.max() <= len(MONTHS), "a value appears more often than there are workbooks"
revised = revisions[(revisions.editions > 1) & (revisions.distinct > 1)].copy()
revised["change_pct"] = ((revised["last"] - revised["first"]) / revised["first"] * 100).round(2)
print(f"{len(revised)} of {int((revisions.editions > 1).sum())} republished values changed between editions")
revised.to_csv(EVIDENCE_DIR / "silver_table1_revisions.csv", index=False)
(revised.groupby(["group", "measure"])
        .agg(months_revised=("reference_month", "size"),
             earliest=("reference_month", "min"), latest=("reference_month", "max"),
             largest_change_pct=("change_pct", lambda c: c.loc[c.abs().idxmax()]))
        .sort_values("months_revised", ascending=False))

106 of 1401 republished values changed between editions


months_revised  \
group            measure                                                                                     
Call durations   Percentage of calls with a duration of 1 to 2 minutes                                  15   
                 Percentage of calls with a duration of 2 to 5 minutes                                  15   
                 Percentage of calls with a duration of over 5 minutes                                  15   
Inbound calls    Rate of inbound calls per 1,000 patients registered at practices included              14   
Call durations   Percentage of calls with a duration of 1 minute or less                                14   
Coverage         Registered patients at included practices                                              13   
                 Patient coverage                                                                        5   
                 Count of practices included                                                             4   
                 Practice coverage [1]                                                                   2   
Call durations   Duration of 1 minute or less                                                            1   
Coverage         Open active practices                                                                   1   
Calls dealt with Calls answered [2]                                                                      1   
                 Calls resulting in a call back request [2]                                              1   
Call durations   Duration of 1 to 2 minutes                                                              1   
Calls dealt with Call backs made                                                                         1   
Call durations   Duration of over 5 minutes                                                              1   
                 Duration of 2 to 5 minutes                                                              1   
Inbound calls    Total Inbound Calls                                                                     1   

                                                                                           earliest  \
group            measure                                                                              
Call durations   Percentage of calls with a duration of 1 to 2 minutes                      2024-10   
                 Percentage of calls with a duration of 2 to 5 minutes                      2024-10   
                 Percentage of calls with a duration of over 5 minutes                      2024-10   
Inbound calls    Rate of inbound calls per 1,000 patients registered at practices included  2024-10   
Call durations   Percentage of calls with a duration of 1 minute or less                    2024-10   
Coverage         Registered patients at included practices                                  2024-11   
                 Patient coverage                                                           2025-07   
                 Count of practices included                                                2024-10   
                 Practice coverage [1]                                                      2024-10   
Call durations   Duration of 1 minute or less                                               2025-11   
Coverage         Open active practices                                                      2025-11   
Calls dealt with Calls answered [2]                                                         2025-11   
                 Calls resulting in a call back request [2]                                 2025-11   
Call durations   Duration of 1 to 2 minutes                                                 2025-11   
Calls dealt with Call backs made                                                            2025-11   
Call durations   Duration of over 5 minutes                                                 2025-11   
                 Duration of 2 to 5 m

**What changed:** most "revisions" are the duration percentages, which turn out to be a
labelling problem (next cell), not a data change. The real revisions are
**registered patients at included practices**, and the rate per 1,000 patients built on
it, restated for back months. The patient denominator for earlier months isn't fixed once
published. Gold uses the August 2026 edition for every month and says so.
`evidence/silver_table1_revisions.csv` has every changed value.

### Do the duration percentages sit against the right labels?

In the August 2026 Table 1, the four call-duration **counts** are labelled ≤1, 1–2, 2–5
and >5 minutes. The four **percentages** under them are labelled 1–2, 2–5, >5 and ≤1
minutes. Different order, and the percentage block puts "1 minute or less" last.

To settle which labels are right, I rebuild both from CBT007 in the durations CSV, which
splits answered calls by call length in seconds. If the counts match under their labels
and the percentages only match when rotated, the percentage labels are wrong in the
published table.

In [16]:
BAND_FROM_LABEL = [("1 minute or less", "le_1min"), ("1 to 2", "1_2min"), ("2 to 5", "2_5min"), ("over 5", "over_5min")]
BAND_SECONDS = {"le_1min": (0, 60), "1_2min": (61, 120), "2_5min": (121, 300), "over_5min": (301, None)}


def band_of(label: str) -> str:
    return next(b for text, b in BAND_FROM_LABEL if text in label)


def duration_check(month: str) -> pd.DataFrame:
    d = pd.read_parquet(SILVER_DIR / "calls_durations" / f"{month}.parquet")
    d = d[d.indicator.eq("CBT007")]
    rebuilt = pd.Series({b: int(d.loc[(d.lower_s >= lo) & ((d.upper_s <= hi) if hi else d.upper_s.isna()), "calls"].sum())
                         for b, (lo, hi) in BAND_SECONDS.items()})
    share = (rebuilt / rebuilt.sum()).round(3)
    t = table1[(table1.publication_month == month) & (table1.reference_month == month)
               & table1.measure.str.contains("uration")].copy()
    t["band"] = t.measure.map(band_of)
    t["kind"] = np.where(t.measure.str.startswith("Percentage"), "share", "count")
    t["rebuilt"] = np.where(t.kind == "count", t.band.map(rebuilt), t.band.map(share))
    t["matches_label"] = (t.value - t.rebuilt).abs() <= np.where(t.kind == "count", 0, 0.0011)
    # which band's rebuilt share does the published % actually equal?
    t["value_actually_is"] = [
        next((b for b in share.index if abs(share[b] - v) <= 0.0011), "no match") if k == "share" else b0
        for v, k, b0 in zip(t.value, t.kind, t.band)]
    return t[["publication_month", "kind", "measure", "band", "value", "rebuilt", "matches_label", "value_actually_is"]]


dur = pd.concat([duration_check(m) for m in MONTHS], ignore_index=True)
dur[dur.publication_month == ANALYSIS_MONTH]

,publication_month,kind,measure,band,value,rebuilt,matches_label,value_actually_is
79,2026-08,count,Duration of 1 minute or less,le_1min,3569721.000,3569721.000,True,le_1min
80,2026-08,count,Duration of 1 to 2 minutes,1_2min,5559164.000,5559164.000,True,1_2min
81,2026-08,count,Duration of 2 to 5 minutes,2_5min,5244761.000,5244761.000,True,2_5min
82,2026-08,count,Duration of over 5 minutes,over_5min,1302130.000,1302130.000,True,over_5min
83,2026-08,share,Percentage of calls with a duration of 1 to 2 minutes,1_2min,0.228,0.355,False,le_1min
84,2026-08,share,Percentage of calls with a duration of 2 to 5 minutes,2_5min,0.355,0.335,False,1_2min
85,2026-08,share,Percentage of calls with a duration of over 5 minutes,over_5min,0.335,0.083,False,2_5min
86,2026-08,share,Percentage of calls with a duration of 1 minute or less,le_1min,0.083,0.228,False,over_5min


In [17]:
by_month = dur.groupby(["publication_month", "kind"]).matches_label.all().unstack()
print("Months where all four duration counts match their labels:", int(by_month["count"].sum()), "of", len(MONTHS))
print("Months where all four duration percentages match their labels:", int(by_month["share"].sum()), "of", len(MONTHS))
by_month

Months where all four duration counts match their labels: 11 of 11
Months where all four duration percentages match their labels: 3 of 11


kind,count,share
publication_month,,
2025-10,True,True
2025-11,True,True
2025-12,True,True
2026-01,True,False
2026-02,True,False
2026-03,True,False
2026-04,True,False
2026-05,True,False
2026-06,True,False


**Finding.** The counts match their labels in all 11 editions. The percentages match
their labels only in the October, November and December 2025 editions. **From the
January 2026 edition onward, every percentage sits one row off.** The published "1 minute
or less" figure (8.3% in August 2026) is really the over-5-minutes share, and the true
1-minute-or-less share is 22.8%, printed against "1 to 2 minutes". Because each edition
republishes the whole series, the error applies to every month back to October 2024 in
those editions. That's why these rows dominate the revisions table above.

Silver keeps the table exactly as published. Gold uses rebuilt shares and cites this
check. That's the kind of thing a reconciliation gate exists to catch.

### Breaks in the series that Gold must respect

- **Wait times, June 2026.** Table 1 note 5: a supplier corrected its wait-time method in
  June 2026, which increased the share answered within 2 minutes. "Data prior to June 2026
  has not been corrected." Wait-time trends across June 2026 aren't like-for-like.
- **Coverage grows month to month.** The Supporting Information page says the rise in
  calls is "largely due to the increase in coverage". Monthly call totals aren't a demand
  trend unless they're normalised by registered patients at included practices.

In [18]:
table1.to_parquet(SILVER_DIR / "published_table1.parquet", index=False)
table2.to_parquet(SILVER_DIR / "published_table2_national_day_time.parquet", index=False)
dur.to_csv(EVIDENCE_DIR / "silver_duration_label_check.csv", index=False)
oc_summary.to_csv(EVIDENCE_DIR / "silver_outcome_sum_check.csv")
practice_match.to_csv(EVIDENCE_DIR / "silver_practice_match.csv", index=False)
print("Silver outputs:")
for p in sorted(SILVER_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(PROJECT_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")

Silver outputs:
  data/silver/.gitkeep  (0.0 MB)
  data/silver/calls_calls_answered/2026-06.parquet  (4.2 MB)
  data/silver/calls_calls_answered/2026-07.parquet  (4.4 MB)
  data/silver/calls_calls_answered/2026-08.parquet  (3.8 MB)
  data/silver/calls_day_time/2025-10.parquet  (7.9 MB)
  data/silver/calls_day_time/2025-11.parquet  (7.6 MB)
  data/silver/calls_day_time/2025-12.parquet  (8.0 MB)
  data/silver/calls_day_time/2026-01.parquet  (8.2 MB)
  data/silver/calls_day_time/2026-02.parquet  (7.5 MB)
  data/silver/calls_day_time/2026-03.parquet  (8.6 MB)
  data/silver/calls_day_time/2026-04.parquet  (8.2 MB)
  data/silver/calls_day_time/2026-05.parquet  (8.0 MB)
  data/silver/calls_day_time/2026-06.parquet  (8.5 MB)
  data/silver/calls_day_time/2026-07.parquet  (8.8 MB)
  data/silver/calls_day_time/2026-08.parquet  (8.1 MB)
  data/silver/calls_durations/2025-10.parquet  (7.3 MB)
  data/silver/calls_durations/2025-11.parquet  (7.0 MB)
  data/silver/calls_durations/2025-12.parquet  (7.4

## What's next — Gold

1. **Reconciliation gate:** reproduce Table 1's August 2026 headlines (inbound, the four
   outcomes, 5,327 / 6,171, patient coverage) and the Monday 08:00–10:00 figure from
   Silver, and show the match.
2. **Coverage by patients:** registered patients behind the 13.7% of practices not
   included, by region, using `practice_dim`.
3. **The remainder:** the 31.9% split into IVR and callback, with the outcome-sum caveat
   from Step 7.
4. **Day × time band grid** (7 × 9) for August 2026, with the bank holiday called out.